# Image compression 


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

# Point this at your PNG (same folder as the notebook, or use an absolute path).
IMAGE_PATH = Path("steve-jobs-iphone.png")

if IMAGE_PATH.exists():
    img = Image.open(IMAGE_PATH)
    print(f"Loaded: {IMAGE_PATH.resolve()}")
else:
    print("image not found!")

#show image
plt.figure(figsize=(8, 8))
plt.imshow(np.asarray(img), cmap="gray" if img.mode == "L" else None)
plt.tight_layout()
plt.show()

In [ ]:
import sys

img_array = np.asarray(img)

on_disk = f"{IMAGE_PATH.stat().st_size:,} bytes" if IMAGE_PATH.is_file() else "n/a"
print("=== Bytes ===")
print(f"PNG on disk:   {on_disk}  ({IMAGE_PATH.resolve()})")
print(f"Pixels in RAM: {img_array.nbytes:,} bytes  |  count: {img_array.size:,}")
print(f"shape {img_array.shape}  dtype {img_array.dtype}  min {img_array.min()}  max {img_array.max()}  mean {img_array.mean():.4f}")

# Imgae Size 
if IMAGE_PATH.is_file():
    tb = IMAGE_PATH.stat().st_size
    print(f"TOTAL PNG file on disk:     {tb:,} bytes ({tb / 1024 / 1024:.2f} MiB)")

np.set_printoptions(linewidth=200, threshold=sys.maxsize)
planes = img_array[np.newaxis, ...] if img_array.ndim == 2 else np.moveaxis(img_array, -1, 0)
for i, plane in enumerate(planes):
    print(f"\n--- channel {i} ---\n{plane}")

# 2D Discrete Cosine Transform (DCT)

The 2D Discrete Cosine Transform converts an image from the **spatial domain**
(pixel values) into the **frequency domain** (cosine coefficients).

Instead of describing the image directly by pixels, the DCT represents the image
as a weighted sum of cosine wave patterns.

---

## 2D DCT Formula

For an image block of size $M \times N$:

$$
C(u,v)=\alpha(u)\alpha(v)
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)
\cos\left(\frac{\pi(2x+1)u}{2M}\right)
\cos\left(\frac{\pi(2y+1)v}{2N}\right)
$$

---

## Basis Function

The cosine basis pattern for frequency pair $(u,v)$ is:

$$
\phi_{u,v}(x,y)=
\alpha(u)\alpha(v)
\cos\left(\frac{\pi(2x+1)u}{2M}\right)
\cos\left(\frac{\pi(2y+1)v}{2N}\right)
$$

Using this notation, the DCT coefficient can also be written as:

$$
C(u,v)=
\sum_{x=0}^{M-1}\sum_{y=0}^{N-1}
f(x,y)\phi_{u,v}(x,y)
$$

---

## Scaling Factors

$$
\alpha(u)=
\begin{cases}
\sqrt{\frac{1}{M}}, & u=0 \\
\sqrt{\frac{2}{M}}, & u>0
\end{cases}
\qquad
\alpha(v)=
\begin{cases}
\sqrt{\frac{1}{N}}, & v=0 \\
\sqrt{\frac{2}{N}}, & v>0
\end{cases}
$$

---

## Meaning of Variables

- $f(x,y)$: pixel value at row $x$, column $y$
- $C(u,v)$: DCT coefficient at frequency $(u,v)$
- $\phi_{u,v}(x,y)$: cosine basis image at frequency $(u,v)$
- $x,y$: pixel coordinates
- $u,v$: frequency indices
- $M,N$: image block dimensions

---

## What $\phi_{u,v}(x,y)$ Means

$\phi_{u,v}(x,y)$ is one building block of the image.

It is a cosine brightness pattern across the block.

Different $(u,v)$ values create different patterns:

- $(0,0)$: constant flat brightness
- small $(u,v)$: slow smooth variation
- large $(u,v)$: rapid alternating detail

Each basis image is orthogonal to the others.

---

## What $C(u,v)$ Means

$C(u,v)$ tells how much of basis pattern $\phi_{u,v}(x,y)$ exists in the image.

Large magnitude:

- strong presence of that pattern

Small magnitude:

- little contribution from that pattern

Positive or negative sign:

- determines phase/orientation contribution

---

## What the DCT Transformation is Capturing

The DCT measures how strongly the image matches each cosine basis pattern.

It projects the image onto a set of cosine waves.

So the transform captures:

- average brightness
- horizontal changes
- vertical changes
- diagonal texture
- smooth shading
- sharp edges
- fine detail

---

## Low vs High Frequencies

- Small $u,v$: low frequency → smooth shading / general structure
- Large $u,v$: high frequency → edges / fine detail / texture

---

## Special Term

$$
C(0,0)
$$

This is the **DC coefficient**.

It represents the average brightness of the image block.

All other coefficients are called **AC coefficients**.

---

## Why DCT is Useful

Most natural images store most energy in low-frequency coefficients.

So many high-frequency coefficients are small and can be removed.

That is why DCT is used in **JPEG image compression**.

---

## Reconstruction

The image can be rebuilt from coefficients:

$$
f(x,y)=
\sum_{u=0}^{M-1}\sum_{v=0}^{N-1}
C(u,v)\phi_{u,v}(x,y)
$$

So the image is a sum of weighted cosine basis patterns.

In [ ]:
from scipy.fft import dctn
import numpy as np
import matplotlib.pyplot as plt

BLOCK_SIZE = 8

rgb = np.asarray(img.convert("RGB"), dtype=np.float64)
h, w, _ = rgb.shape
N = BLOCK_SIZE
hp, wp = (h + N - 1) // N * N, (w + N - 1) // N * N
Hb, Wb = hp // N, wp // N

channel_names = ("R", "G", "B")
pads, dct_layouts, dc_maps, dct_logmag, dct_blocks_all = [], [], [], [], []

for c in range(3):
    plane = rgb[:, :, c]

    # pad channel so height/width are multiples of 8
    pad = np.zeros((hp, wp), dtype=np.float64)
    pad[:h, :w] = plane

    # block DCT storage: [block_row, block_col, u, v]
    dct_blocks = np.zeros((Hb, Wb, N, N), dtype=np.float64)

    for bi in range(Hb):
        for bj in range(Wb):
            blk = pad[bi * N:(bi + 1) * N, bj * N:(bj + 1) * N]
            dct_blocks[bi, bj] = dctn(blk - 128.0, norm="ortho")

    # tile blocks back into one large array for visualization
    dct_layout = np.block([
        [dct_blocks[bi, bj] for bj in range(Wb)]
        for bi in range(Hb)
    ])

    # DC coefficient from each block
    dc_map = dct_blocks[:, :, 0, 0]

    # log magnitude for display
    logmag = np.log1p(np.abs(dct_layout[:h, :w]))

    pads.append(pad)
    dct_layouts.append(dct_layout)
    dc_maps.append(dc_map)
    dct_logmag.append(logmag)
    dct_blocks_all.append(dct_blocks)

print("=== DCT on each RGB channel separately ===")
print(f"Image: {h}x{w}")
print(f"Block size: {N}x{N}")
print(f"Block grid: {Hb} x {Wb}")
print(f"Coefficients per block per channel: {N*N}")

# print every transformed block
with np.printoptions(precision=3, suppress=True, linewidth=160):
    for c, name in enumerate(channel_names):
        print(f"\n==================== Channel {name} ====================")
        dct_blocks = dct_blocks_all[c]

        for bi in range(Hb):
            for bj in range(Wb):
                print(f"\nBlock ({bi}, {bj}) DCT coefficients:")
                print(dct_blocks[bi, bj])

                dc = dct_blocks[bi, bj, 0, 0]
                print(f"DC = {dc:.3f}")

fig, axes = plt.subplots(3, 3, figsize=(15, 10))
fig.suptitle("RGB: DCT per channel (middle column = log1p(|DCT|) so AC coeffs are visible)", y=1.01)

for c, name in enumerate(channel_names):
    axes[c, 0].imshow(pads[c][:h, :w], cmap="gray", vmin=0, vmax=255)
    axes[c, 0].set_ylabel(f"{name}", rotation=0, labelpad=20, fontsize=12, fontweight="bold")
    axes[c, 0].set_title("pixels" if c == 0 else "")
    axes[c, 0].axis("off")

    lm = dct_logmag[c]
    vmax = float(np.percentile(lm, 99.5)) + 1e-9
    im1 = axes[c, 1].imshow(lm, cmap="magma", vmin=0.0, vmax=vmax)
    axes[c, 1].set_title("log1p(|DCT|) tiled" if c == 0 else "")
    axes[c, 1].axis("off")
    plt.colorbar(im1, ax=axes[c, 1], fraction=0.046)

    dm = dc_maps[c]
    dv = np.max(np.abs(dm)) + 1e-9
    im2 = axes[c, 2].imshow(dm, cmap="coolwarm", vmin=-dv, vmax=dv)
    axes[c, 2].set_title("DC map" if c == 0 else "")
    axes[c, 2].axis("off")
    plt.colorbar(im2, ax=axes[c, 2], fraction=0.046)

plt.tight_layout()
plt.show()


## Output Interpretation

The figure contains **3 rows × 3 columns**.

Each row corresponds to one RGB channel:

- **R** = Red channel  
- **G** = Green channel  
- **B** = Blue channel  

---

## Column 1 — Pixel Intensities

This column shows the original image, but only one color channel at a time, displayed in grayscale.

- White = high intensity of that channel
- Black = low intensity of that channel

Examples:

- In the **Red** row, bright regions contain strong red content.
- In the **Green** row, bright regions contain strong green content.
- In the **Blue** row, bright regions contain strong blue content.

This is not the full color image. It is each RGB channel isolated and visualized in grayscale.

This is the image in the **spatial domain**.

---

## Column 2 — log1p(|DCT|) Tiled

This column shows the DCT coefficient magnitudes for every 8×8 block.

Each 8×8 image block is transformed into an 8×8 frequency block, and all blocks are tiled back into image layout.

The displayed values are:

$$
\log(1 + |C(u,v)|)
$$

We use log scale because:

- DC coefficients are usually very large.
- Many AC coefficients are small.
- Log scaling makes both strong and weak coefficients visible at the same time.

### Bright Spots Mean Strong Frequency Energy

A bright spot means that cosine frequency pattern has a large coefficient.

This usually indicates:

- strong brightness/color change inside that block
- edges
- texture
- repeated patterns
- fine detail

### Dark Areas Mean Weak Frequency Energy

This indicates:

- smooth regions
- flat color
- little local variation

### Inside Each 8×8 Tile

- Top-left = DC coefficient (average brightness of block)
- Near top-left = low frequencies (smooth shading / gradual transitions)
- Farther away = higher frequencies (rapid pixel changes)

Bright only near top-left:

- smooth block with little detail

Bright spread across tile:

- textured region, sharp edges, detailed structure

Bright bottom-right:

- rapid alternating pixel changes, noise, fine texture

This column shows the image in the **frequency domain**.

---

## Column 3 — DC Map

This column shows only the DC coefficient from each 8×8 block:

$$
C(0,0)
$$

Each block contributes one value, producing a low-resolution brightness map.

Large positive values:

- brighter-than-average blocks

Large negative values:

- darker-than-average blocks

Values near zero:

- medium brightness blocks

This preserves coarse image structure but removes edges and texture.

---

## Main Interpretation

These three columns show:

1. Original RGB channels shown individually in grayscale  
2. Local frequency energy of each 8×8 block  
3. Average brightness of each block

This demonstrates why JPEG compression works:

- most visual energy is concentrated in low-frequency coefficients
- many high-frequency coefficients are small
- smooth regions require very little data

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

STEP_SIZE = 25.0
channel_names = ("R", "G", "B")

# Quantize first in float, check range, then choose a safe integer type
q_float = [np.round(dct_blocks_all[c] / STEP_SIZE) for c in range(3)]

global_min = min(int(q.min()) for q in q_float)
global_max = max(int(q.max()) for q in q_float)

if -128 <= global_min and global_max <= 127:
    q_dtype = np.int8
elif -32768 <= global_min and global_max <= 32767:
    q_dtype = np.int16
else:
    q_dtype = np.int32

print(f"Quantized value range: [{global_min}, {global_max}]")
print(f"Using dtype: {q_dtype.__name__}")

quant_rgb = []
quant_layouts = []

for c in range(3):
    Q = q_float[c].astype(q_dtype)
    quant_rgb.append(Q)

    q_layout = np.block([
        [Q[bi, bj] for bj in range(Wb)]
        for bi in range(Hb)
    ])
    quant_layouts.append(q_layout)

print("\n=== Quantized DCT ===")
print(f"STEP_SIZE = {STEP_SIZE}")
print("Q = round(DCT / STEP_SIZE)\n")

for c, name in enumerate(channel_names):
    Q = quant_rgb[c]
    nz = np.count_nonzero(Q)

    print(f"==================== {name} Channel ====================")
    print(f"shape: {Q.shape}")
    print(f"dtype: {Q.dtype}")
    print(f"nonzero: {nz:,} / {Q.size:,}")
    print(f"min/max: {Q.min()} / {Q.max()}")
    print("\nFull tiled quantized coefficient matrix:\n")

    with np.printoptions(linewidth=220, threshold=np.inf):
        print(quant_layouts[c])

    print("\n")

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Quantized DCT (Higher Contrast)", fontsize=16)

for c, name in enumerate(channel_names):
    q = quant_layouts[c].astype(np.float64)

    disp = np.sign(q) * np.log1p(np.abs(q))
    m = np.percentile(np.abs(disp), 97) + 1e-9

    im = axes[c].imshow(
        disp,
        cmap="RdBu_r",
        vmin=-m,
        vmax=m
    )
    axes[c].set_title(f"{name} channel")
    axes[c].axis("off")
    plt.colorbar(im, ax=axes[c], fraction=0.046)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Zero Mask (white = 0)", y=1.03)

for c, name in enumerate(channel_names):
    zero_mask = (quant_layouts[c] == 0)
    axes[c].imshow(zero_mask, cmap="gray")
    axes[c].set_title(f"{name} channel")
    axes[c].axis("off")

plt.tight_layout()
plt.show()

## Quantized DCT Interpretation

This section visualizes the DCT coefficients **after quantization**.

For each 8×8 block, every DCT coefficient is converted to an integer using:

$$
Q(u,v)=\operatorname{round}\left(\frac{C(u,v)}{25}\right)
$$

where:

- \(C(u,v)\) is the original DCT coefficient
- \(25\) is the quantization step size
- \(Q(u,v)\) is the stored quantized value

This means each coefficient is divided by 25 and rounded.

Examples:

$$
100 \rightarrow 4,\quad
12 \rightarrow 0,\quad
-63 \rightarrow -3
$$

---

## What Quantization Does

Quantization reduces precision.

Small coefficients become zero, while larger coefficients remain nonzero.

This removes weak frequency components and keeps stronger ones.

In many image blocks:

- low-frequency coefficients are larger
- high-frequency coefficients are smaller

So quantization often preserves coarse structure while removing fine detail.

This is the main lossy compression step used in JPEG.

---

## First Figure — Quantized DCT Coefficients

This figure shows all quantized coefficients for every block in each RGB channel.

Each small 8×8 tile corresponds to one image block.

Inside each tile:

- top-left = DC coefficient (average brightness)
- near top-left = low-frequency terms
- farther away = higher-frequency terms

Color meaning:

- red = positive coefficient
- blue = negative coefficient
- white/light = near zero

Bright colored regions indicate stronger remaining frequency energy.

White regions indicate coefficients that became zero or very small after quantization.

---

## Second Figure — Zero Mask

This figure shows where coefficients became zero after quantization.

- White = zero coefficient
- Black = nonzero coefficient

Large white areas indicate many discarded coefficients.

This demonstrates why compression works well:

- smooth regions need only a few low-frequency coefficients
- many high-frequency terms can be removed

---

## Main Interpretation

Quantization keeps the most important frequency information and discards weaker detail.

The larger the step size:

- more zeros
- stronger compression
- lower image quality

The smaller the step size:

- fewer zeros
- better quality
- larger file size

In [ ]:
from pathlib import Path
import numpy as np
from scipy import sparse

# After quantization, many DCT coefficients become exactly 0.
# Use sparse storage: keep mainly nonzero values + sparse indices.
# This is lossless for the quantized data and works well for arrays
# with many zeros.

out_path = Path("steve_jobs_compressed_sparse.npz")

total_coeffs = 0
total_nonzeros = 0

for c in range(3):
    total_coeffs += quant_rgb[c].size
    total_nonzeros += np.count_nonzero(quant_rgb[c])

total_zeros = total_coeffs - total_nonzeros

Qr = sparse.csr_matrix(quant_rgb[0].reshape(1, -1))
Qg = sparse.csr_matrix(quant_rgb[1].reshape(1, -1))
Qb = sparse.csr_matrix(quant_rgb[2].reshape(1, -1))

np.savez_compressed(
    out_path,
    Qr_data=Qr.data, Qr_indices=Qr.indices, Qr_indptr=Qr.indptr, Qr_shape=Qr.shape,
    Qg_data=Qg.data, Qg_indices=Qg.indices, Qg_indptr=Qg.indptr, Qg_shape=Qg.shape,
    Qb_data=Qb.data, Qb_indices=Qb.indices, Qb_indptr=Qb.indptr, Qb_shape=Qb.shape,
    h=np.int32(h), w=np.int32(w), N=np.int32(N),
    Hb=np.int32(Hb), Wb=np.int32(Wb), STEP_SIZE=np.float32(STEP_SIZE),
)

# print("=== Quantized Coefficient Statistics ===")
# print(f"Total coefficients : {total_coeffs:,}")
# print(f"Nonzero values     : {total_nonzeros:,}")
# print(f"Zero values        : {total_zeros:,}")
# print(f"Zero percentage    : {100 * total_zeros / total_coeffs:.2f}%")
# print()

# print("Saved sparse coefficient files:")
# print(Path("Qr_sparse.npz").resolve(), Path("Qr_sparse.npz").stat().st_size, "bytes")
# print(Path("Qg_sparse.npz").resolve(), Path("Qg_sparse.npz").stat().st_size, "bytes")
# print(Path("Qb_sparse.npz").resolve(), Path("Qb_sparse.npz").stat().st_size, "bytes")

# print("\nSaved metadata file:")
# print(out_path.resolve(), out_path.stat().st_size, "bytes")

# total_size =
# (
#     Path("Qr_sparse.npz").stat().st_size +
#     Path("Qg_sparse.npz").stat().st_size +
#     Path("Qb_sparse.npz").stat().st_size +
#     out_path.stat().st_size
# )

# print(f"\nTotal stored size: {total_size:,} bytes")


In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
from scipy.fft import idctn
from PIL import Image
from io import BytesIO

#reload coefficients
meta_path = Path("steve_jobs_compressed_sparse.npz")

meta = np.load(meta_path)

h2 = int(meta["h"])
w2 = int(meta["w"])
N2 = int(meta["N"])
Hb2 = int(meta["Hb"])
Wb2 = int(meta["Wb"])
STEP2 = float(meta["STEP_SIZE"])


def load_csr(f, prefix):
    return sparse.csr_matrix(
        (f[f"{prefix}_data"], f[f"{prefix}_indices"], f[f"{prefix}_indptr"]),
        shape=f[f"{prefix}_shape"]
    )

with np.load(meta_path) as f:
    Qr = load_csr(f, "Qr").toarray().reshape(Hb2, Wb2, N2, N2)
    Qg = load_csr(f, "Qg").toarray().reshape(Hb2, Wb2, N2, N2)
    Qb = load_csr(f, "Qb").toarray().reshape(Hb2, Wb2, N2, N2)



In [ ]:
# dequantize each block

hp2 = Hb2 * N2
wp2 = Wb2 * N2

recon_pad = np.zeros((3, hp2, wp2), dtype=np.float64)

for c, Q in enumerate((Qr, Qg, Qb)):
    for bi in range(Hb2):
        for bj in range(Wb2):
            dq = Q[bi, bj].astype(np.float64) * STEP2
            recon_pad[c,
                      bi * N2:(bi + 1) * N2,
                      bj * N2:(bj + 1) * N2] = idctn(dq, norm="ortho") + 128.0

recon_sparse = np.stack(
    [np.clip(recon_pad[c, :h2, :w2], 0.0, 255.0) for c in range(3)],
    axis=-1
).astype(np.uint8)

plt.figure(figsize=(9, 7))
plt.imshow(recon_sparse)
plt.title("Reconstructed from Sparse Compressed Files")
plt.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
img_array = np.asarray(img)

on_disk = IMAGE_PATH.stat().st_size if IMAGE_PATH.is_file() else None
on_disk_str = f"{on_disk:,} bytes" if on_disk is not None else "n/a"

# Total DCT coefficients before quantization/compression
before_coeffs = 3 * Hb * Wb * N * N

# After compression:
# nonzero quantized coefficients kept in sparse form
after_coeffs = 0
for c in range(3):
    after_coeffs += np.count_nonzero(quant_rgb[c])

print("=== Original Image ===")
print(f"PNG on disk:   {on_disk_str}  ({IMAGE_PATH.resolve()})")
print(f"Pixels in RAM: {img_array.nbytes:,} bytes")
print(f"shape         : {img_array.shape}")
print(f"dtype         : {img_array.dtype}")


# -----------------------------------------
# compressed file sizes
# -----------------------------------------

from pathlib import Path

meta_file = Path("steve_jobs_compressed_sparse.npz")
compressed_total = meta_file.stat().st_size if meta_file.is_file() else 0

print("\n=== Coefficient Comparison ===")
print(f"Before compression coeffs : {before_coeffs:,}")
print(f"After compression coeffs  : {after_coeffs:,}")
print(f"Removed coeffs            : {before_coeffs - after_coeffs:,}")
print(f"Zeroed percentage         : {100 * (before_coeffs - after_coeffs) / before_coeffs:.2f}%")

print("\n=== Storage Comparison ===")
print(f"Compressed total size     : {compressed_total:,} bytes")

if on_disk is not None:
    print(f"Original PNG size         : {on_disk:,} bytes")

    diff = on_disk - compressed_total

    if diff > 0:
        print(f"Space saved              : {diff:,} bytes")
        print(f"Compression ratio        : {on_disk / compressed_total:.2f}x smaller")
    else:
        print(f"Larger than original     : {-diff:,} bytes")
        print(f"Size ratio               : {compressed_total / on_disk:.2f}x larger")

